# Code 8.6.4: Assembling a retrieval-augmented prompt

Retrieves the two most similar documents for a question with the sentence embedding model and collection of Code 7.7.4, and builds the prompt shown in the text. It needs `sentence-transformers`, which downloads the model on first use. The prompt would then be passed to a model for generation, as in Code 8.6.1.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
docs = [
    "The cat sat on the windowsill, watching birds.",
    "Our quarterly revenue grew by twelve percent.",
    "How to reset a forgotten email password.",
    "Kittens need to be fed several times a day.",
    "The central bank raised interest rates again.",
    "Steps for recovering access to your account.",
]
doc_emb = model.encode(docs, normalize_embeddings=True)       # (6, 384), unit length

def search(query, k=2):
    q = model.encode(query, normalize_embeddings=True)
    scores = doc_emb @ q                                      # cosine similarities
    top = scores.argsort()[::-1][:k]
    return [(float(scores[i]), docs[i]) for i in top]

def build_prompt(question, chunks):
    context = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(chunks))
    return (
        "Answer the question using only the numbered passages below. "
        "Cite the passages you use, and say so if they do not contain the answer.\n\n"
        f"{context}\n\nQuestion: {question}\nAnswer:"
    )

chunks = [doc for _, doc in search("I can't log in to my mailbox", k=2)]
print(build_prompt("I can't log in to my mailbox. What should I do?", chunks))
